In [1]:
# === STEP 1: Import Libraries & Connect to Database ===
import sqlite3
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt

In [2]:
conn = sqlite3.connect("stackoverflow.db")

In [3]:
# === STEP 2: Get Monthly Post Counts via SQL Aggregation ===
posts_query = """
    SELECT strftime('%Y-%m', CreationDate) AS YearMonth, COUNT(*) AS PostCount
    FROM posts
    WHERE CreationDate IS NOT NULL
    GROUP BY YearMonth
    ORDER BY YearMonth
"""
monthly_posts = pd.read_sql_query(posts_query, conn)

In [ ]:
monthly_posts.info()

In [ ]:
monthly_posts.head()

In [ ]:
# === STEP 3: Get Monthly Comment Counts via SQL Aggregation ===
comments_query = """
    SELECT strftime('%Y-%m', CreationDate) AS YearMonth, COUNT(*) AS CommentCount
    FROM comments
    WHERE CreationDate IS NOT NULL
    GROUP BY YearMonth
    ORDER BY YearMonth
"""
monthly_comments = pd.read_sql_query(comments_query, conn)

In [ ]:
monthly_comments = info()

In [ ]:
monthly_comments = head()

In [ ]:
# === STEP 4: Merge the Two Datasets on YearMonth ===
engagement_df = pd.merge(monthly_posts, monthly_comments, on="YearMonth", how="outer").fillna(0)

In [ ]:
engagement_df.head()

In [ ]:
# Optional: Make sure counts are integers
engagement_df["PostCount"] = engagement_df["PostCount"].astype(int)
engagement_df["CommentCount"] = engagement_df["CommentCount"].astype(int)

In [ ]:
# === STEP 5: Create Grouped Bar Plot for Posts & Comments ===
plt.figure(figsize=(14, 7))

# Melt data for sns.barplot with hue
engagement_melted = engagement_df.melt(id_vars="YearMonth", value_vars=["PostCount", "CommentCount"],
                                       var_name="ActivityType", value_name="Count")

ax = sns.barplot(data=engagement_melted, x="YearMonth", y="Count", hue="ActivityType", palette="viridis")

plt.title("User Engagement per Year-Month (Posts vs Comments)")
plt.xlabel("Year-Month")
plt.ylabel("Count")
plt.xticks(rotation=90, ha='right')
plt.legend(title="Activity Type")

# Clean x-axis labels (grouped by year)
year_month_pattern = []
unique_years = engagement_df["YearMonth"].str.split("-").str[0].unique()

for year in unique_years:
    year_month_pattern.append(year)
    year_month_pattern.extend([' '] * (engagement_df["YearMonth"].str.startswith(year).sum() - 1))

ax.set_xticklabels(year_month_pattern)

plt.tight_layout()
plt.show()
